# Auto-Quant - TFLite Conversion (Google Colab)

Converts a HuggingFace vision model to TFLite in multiple formats:
- FP32 — full precision baseline
- FP16 — half precision, best for high-end mobile GPUs
- Dynamic range — weights-only INT8, no calibration needed
- INT8 — full integer quantization, best for mid-range devices
- INT16 — for models sensitive to INT8 accuracy loss

**Instructions:**
1. Runtime -> Change runtime type -> CPU
2. Run Cell 1 only
3. Runtime -> Restart session
4. Run all remaining cells

**Output:** All .tflite files auto-downloaded at end.

In [ ]:
# =============================================================================
# CELL 1 - Run FIRST, then restart runtime before running anything else
# Do NOT install tensorflow or numpy here - use Colab pre-installed versions
# =============================================================================
!pip install -q onnx onnxruntime onnx2tf sng4onnx

import numpy as np
print(f"numpy version: {np.__version__}")
print("If no errors above, restart runtime and run from Cell 2")

In [ ]:
# =============================================================================
# CELL 2 - Configuration
# =============================================================================
MODEL_ID   = "google/mobilenet_v2_1.0_224"  # HuggingFace vision model repo ID
IMAGE_SIZE = 224                             # Input image size
HF_TOKEN   = None                            # Set if model is gated
OUTPUT_DIR = "/content/tflite_output"

In [ ]:
# =============================================================================
# CELL 3 - Verify environment
# =============================================================================
import tensorflow as tf
import torch
import numpy as np
import onnx
import onnx2tf

print(f"numpy:      {np.__version__}")
print(f"tensorflow: {tf.__version__}")
print(f"torch:      {torch.__version__}")
print(f"onnx:       {onnx.__version__}")
print(f"onnx2tf:    {onnx2tf.__version__}")
print("Environment ready.")

In [ ]:
# =============================================================================
# CELL 4 - Download model from HuggingFace
# =============================================================================
from transformers import AutoModelForImageClassification, AutoConfig
import torch

print(f"Loading: {MODEL_ID}")
cfg = AutoConfig.from_pretrained(MODEL_ID, token=HF_TOKEN)
image_size = getattr(cfg, "image_size", IMAGE_SIZE)

model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    token=HF_TOKEN,
).eval()

print(f"Model loaded. Image size: {image_size}x{image_size}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# =============================================================================
# CELL 5 - Export to ONNX
# =============================================================================
import os
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
onnx_path = output_dir / "model.onnx"

print("Exporting to ONNX...")
dummy_input = torch.randn(1, 3, image_size, image_size)

torch.onnx.export(
    model,
    dummy_input,
    str(onnx_path),
    input_names=["input"],
    output_names=["output"],
    opset_version=11,
)

size_mb = onnx_path.stat().st_size / (1024 * 1024)
print(f"ONNX saved: {onnx_path} ({size_mb:.1f} MB)")

In [ ]:
# =============================================================================
# CELL 6 - Convert ONNX to TFLite (all formats) via onnx2tf + TF converter
# =============================================================================
import onnx2tf
import numpy as np
import tensorflow as tf
import shutil
from pathlib import Path

onnx2tf_out = output_dir / "onnx2tf_out"
if onnx2tf_out.exists():
    shutil.rmtree(onnx2tf_out)

# Generate calibration data for INT8
calib_path = output_dir / "calib_input.npy"
calib_data = np.random.rand(100, 3, image_size, image_size).astype(np.float32)
np.save(str(calib_path), calib_data)
print(f"Calibration data: {calib_data.shape}")

# Run onnx2tf -- produces FP32, FP16, INT8 flatbuffers directly
print("Converting via onnx2tf...")
onnx2tf.convert(
    input_onnx_file_path=str(onnx_path),
    output_folder_path=str(onnx2tf_out),
    output_integer_quantized_tflite=True,
    quant_type="per-tensor",
    custom_input_op_name_np_data_path=[
        ["input", str(calib_path), 0.0, 1.0]
    ],
    non_verbose=True,
)

print("\nRaw files from onnx2tf:")
for f in sorted(onnx2tf_out.rglob("*.tflite")):
    print(f"  {f.name} ({f.stat().st_size / 1024**2:.1f} MB)")

# Copy FP32 and FP16 from onnx2tf output
fp32_src = next(onnx2tf_out.rglob("*float32*.tflite"))
fp16_src = next(onnx2tf_out.rglob("*float16*.tflite"))
int8_candidates = (
    list(onnx2tf_out.rglob("*integer_quant*.tflite")) +
    list(onnx2tf_out.rglob("*int8*.tflite")) +
    list(onnx2tf_out.rglob("*quant*.tflite"))
)

fp32_path = output_dir / "model_fp32.tflite"
fp16_path = output_dir / "model_fp16.tflite"
shutil.copy(fp32_src, fp32_path)
shutil.copy(fp16_src, fp16_path)
print(f"\nFP32: {fp32_path} ({fp32_path.stat().st_size / 1024**2:.1f} MB)")
print(f"FP16: {fp16_path} ({fp16_path.stat().st_size / 1024**2:.1f} MB)")

if int8_candidates:
    int8_path = output_dir / "model_int8.tflite"
    shutil.copy(int8_candidates[0], int8_path)
    print(f"INT8: {int8_path} ({int8_path.stat().st_size / 1024**2:.1f} MB)")
else:
    print("INT8 not produced by onnx2tf directly, quantizing from FP32...")
    int8_path = None

# Dynamic range quantization (weights-only, no calibration needed)
print("\nGenerating dynamic range quantization...")
converter = tf.lite.TFLiteConverter.from_saved_model
# Load FP32 flatbuffer and apply dynamic range quant
# onnx2tf SavedModel path if available, else skip
saved_model_candidates = list(onnx2tf_out.rglob("saved_model.pb"))
if saved_model_candidates:
    saved_model_dir = saved_model_candidates[0].parent
    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    dynamic_model = converter.convert()
    dynamic_path = output_dir / "model_dynamic.tflite"
    dynamic_path.write_bytes(dynamic_model)
    print(f"Dynamic: {dynamic_path} ({dynamic_path.stat().st_size / 1024**2:.1f} MB)")

    # INT16 quantization
    print("Generating INT16 quantization...")
    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    def rep_dataset_int16():
        for i in range(100):
            yield [calib_data[i:i+1].transpose(0, 2, 3, 1)]

    converter.representative_dataset = rep_dataset_int16
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.EXPERIMENTAL_TFLITE_BUILTINS_ACTIVATIONS_INT16_WEIGHTS_INT8
    ]
    int16_model = converter.convert()
    int16_path = output_dir / "model_int16.tflite"
    int16_path.write_bytes(int16_model)
    print(f"INT16: {int16_path} ({int16_path.stat().st_size / 1024**2:.1f} MB)")
else:
    print("No SavedModel found - skipping dynamic range and INT16 (flatbuffer_direct mode)")
    print("FP32, FP16, INT8 are available from onnx2tf output")

In [ ]:
# =============================================================================
# CELL 7 - Summary and download
# =============================================================================
import json
from google.colab import files
from pathlib import Path

output_dir = Path(OUTPUT_DIR)

tflite_files = sorted(output_dir.glob("*.tflite"))

print("Output files:")
results = {}
for f in tflite_files:
    size_mb = f.stat().st_size / 1024**2
    print(f"  {f.name:<35} {size_mb:.1f} MB")
    results[f.stem] = {"path": str(f), "size_mb": round(size_mb, 1)}

meta = {
    "model_id": MODEL_ID,
    "image_size": image_size,
    "backend": "onnx2tf",
    "conversion_path": "pytorch -> onnx (opset 11) -> tflite",
    "outputs": results,
}
meta_path = output_dir / "autoquant_meta.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("\nDownloading...")
for f in tflite_files:
    files.download(str(f))
files.download(str(meta_path))
print("Done.")